# Convert one PIVview result

Run from an editable source checkout to use the bundled inputs. A snapshot is one
processed image-pair result. This example uses 2D2C data, adds standard names and
validates metadata. The recording date below is illustrative; replace it with the
actual acquisition time for your measurements.

In [ ]:
from datetime import datetime
from pathlib import Path
import h5py
from piv2hdf import PIVSnapshot, set_config, tutorial
from piv2hdf.pivview import PIVViewNcFile
from piv2hdf.pivview.user_operations import add_standard_name_operation

source = tutorial.PIVview.get_snapshot_nc_files()[0]
reader = PIVViewNcFile(source, parameter=tutorial.PIVview.get_parameter_file(),
                       udo=add_standard_name_operation)
snapshot = PIVSnapshot(reader, recording_dtime=datetime(2026, 1, 1, 12))

## Write a new output

`creator` is required by the selected convention. Use your own name and metadata
for real data. The output goes into `outputs/` relative to this notebook. A
successful rerun replaces it; use a different filename to retain earlier results.

In [ ]:
with set_config(convention="planar_piv"):
    output = snapshot.to_hdf(
        hdf_filename=Path("outputs/pivview_snapshot.hdf"),
        piv_attributes={"creator": {"name": "Example researcher"}, "piv_medium": "air"},
    )
print(output)

## Inspect values and units

`u` and `v` have shape `(ny, nx)`. Invalid vectors become NaN. `valid` describes
the processed field; `raw_u` and `raw_piv_flags` retain the unmasked source values.
A single snapshot has no time-series mean. See the [plane example](02_plane_pivview.ipynb)
for timing and statistics.

In [ ]:
with h5py.File(output, "r") as data:
    print("u:", data["u"].shape, data["u"].attrs["units"])
    print("valid vectors:", int(data["valid"][...].sum()))
    print("recording time:", data["time"][()])